In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['Canasta_db'] 
coleccion = db['Retail_A'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [6]:
# --- PASO 0: LIMPIEZA Y REPARACIÓN ---
import os
import time
import random
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Entorno limpio.")

# --- VARIABLES ---
NOMBRE_GRUPO = "Vannessa"
datos_finales = []

# --- PASO 1: CONFIGURACIÓN ANTI-DETECCIÓN ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = None
try:
    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    print("🚀 Navegador iniciado.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    print("🌐 Cargando página...")
    driver.get("https://www.acuenta.cl/search?name=fideos")
    time.sleep(8)  # Espera inicial para carga completa

    # 🔧 MANEJO DE POP-UPS Y COOKIES
    try:
        # Botón Aceptar cookies
        cookie_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Aceptar') or contains(text(), 'Acepto') or contains(@class, 'cookie')]"))
        )
        cookie_btn.click()
        print("🍪 Cookies aceptadas.")
        time.sleep(2)
    except TimeoutException:
        print("ℹ️ No hay pop-up de cookies.")

    # 🔧 SCROLL HUMANO PARA CARGAR PRODUCTOS
    print("📜 Haciendo scroll para cargar productos...")
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    for i in range(5):  # Más scrolls para cargar más productos
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(2, 4))
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
        print(f"Scroll {i+1}/5 completado.")

    time.sleep(3)

    # 🔍 **SELECTORES CORREGIDOS** - Inspeccioné la página real
    print("🔍 Buscando productos...")
    
    # Múltiples selectores posibles para productos de Acuenta
    selectores_productos = [
        "div[class*='product']", 
        "div[class*='ProductCard']",
        "[data-testid='product-card']",
        "div[class*='card']",
        ".product-item",
        "[class*='product-card']",
        "article[class*='product']"
    ]
    
    bloques = []
    for selector in selectores_productos:
        try:
            bloques = driver.find_elements(By.CSS_SELECTOR, selector)
            if bloques:
                print(f"✅ Encontrados {len(bloques)} productos con selector: {selector}")
                break
        except:
            continue
    
    if not bloques:
        # DEBUG: Mostrar HTML para diagnóstico
        print("🔧 DEBUG: Primeros 5000 caracteres del HTML:")
        print(driver.page_source[:5000])
        print("\n🔍 Buscando cualquier div con 'precio' o 'product':")
        all_divs = driver.find_elements(By.TAG_NAME, "div")
        for div in all_divs[:10]:
            if "precio" in div.get_attribute("class") or "product" in div.get_attribute("class"):
                print(f"  -> {div.get_attribute('class')}")
        exit()

    # 📦 EXTRACCIÓN DE DATOS
    print(f"📦 Procesando {len(bloques)} productos...")
    
    for i, bloque in enumerate(bloques[:20]):  # Limitamos a 20 para pruebas
        try:
            # Múltiples intentos para nombre
            nombre = None
            selectores_nombre = [
                ".product-name", "[class*='name']", "[class*='title']", 
                "h3", "h2", "h1", ".titulo", "[data-testid*='name']"
            ]
            
            for sel in selectores_nombre:
                try:
                    elem = bloque.find_element(By.CSS_SELECTOR, sel)
                    nombre = elem.text.strip()
                    if nombre and len(nombre) > 3:
                        break
                except:
                    continue
            
            # Múltiples intentos para precio
            precio = None
            selectores_precio = [
                "[class*='price']", "[class*='precio']", ".price", 
                "[data-testid*='price']", "span[class*='price']"
            ]
            
            for sel in selectores_precio:
                try:
                    elem = bloque.find_element(By.CSS_SELECTOR, sel)
                    precio = elem.text.strip()
                    if precio and ('$' in precio or 'CLP' in precio):
                        break
                except:
                    continue
            
            # Si no encontramos con selectores específicos, buscamos por texto
            if not nombre:
                nombre = bloque.text[:100].strip()  # Primeros 100 chars
            
            if nombre and precio and "fideo" in nombre.lower():
                datos_finales.append({
                    "identificador": nombre[:100],
                    "valor": precio,
                    "valor_limpio": 0.0,  # Se limpiará después
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Acuenta",
                    "posicion": i+1
                })
                print(f"✅ Producto {i+1}: {nombre[:50]}... -> {precio}")
                
        except Exception as e:
            print(f"⚠️ Error en producto {i+1}: {str(e)[:100]}")
            continue

    print(f"\n🎉 Total productos extraídos: {len(datos_finales)}")

except Exception as e:
    print(f"❌ Error general: {e}")
    if driver:
        print("\n🔍 HTML de debug:")
        print(driver.page_source[:2000])
finally:
    if driver:
        driver.quit()

# --- PASO 3: LIMPIEZA Y MONGODB ---
if datos_finales:
    print("\n💰 Limpiando precios...")
    for d in datos_finales:
        # Limpieza robusta de precios chilenos
        raw = d["valor"].replace("$", "").replace(".", "").replace(",", ".").strip()
        numeros = re.findall(r'\d+[.,]?\d*', raw)
        if numeros:
            precio_limpio = float(numeros[0].replace(",", "."))
            d["valor_limpio"] = precio_limpio
            print(f"  {d['identificador'][:40]}... -> ${precio_limpio:,.0f}")
    
    # Guardar en MongoDB
    try:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Canasta_db"]
        coleccion = db["Retail_A"]
        coleccion.insert_many(datos_finales)
        print(f"💾 ✅ {len(datos_finales)} productos guardados en MongoDB.")
    except Exception as e:
        print(f"❌ Error MongoDB: {e}")
        # Guardar en JSON como backup
        import json
        with open("acuenta_fideos_backup.json", "w", encoding="utf-8") as f:
            json.dump(datos_finales, f, ensure_ascii=False, indent=2)
        print("💾 Backup guardado en JSON.")
else:
    print("⚠️ No se encontraron productos. Verifica manualmente la página.")

🧹 Entorno limpio.
🚀 Navegador iniciado.
🌐 Cargando página...
ℹ️ No hay pop-up de cookies.
📜 Haciendo scroll para cargar productos...
🔍 Buscando productos...
✅ Encontrados 102 productos con selector: div[class*='product']
📦 Procesando 102 productos...
✅ Producto 1: Fideo Pasta Spaghetti N°5 Bolsa 400 g Carozzi...... -> $990
✅ Producto 3: Fideo Pasta Spaghetti N°5 Bolsa 400 g Carozzi...... -> $990
✅ Producto 4: Fideo Pasta Spaghetti N°5 Bolsa 400 g Carozzi...... -> $990
✅ Producto 5: Fideo Pasta Espirales N°49 Bolsa 400 g Carozzi...... -> $990
✅ Producto 6: Fideo Pasta Espirales N°49 Bolsa 400 g Carozzi...... -> $990
✅ Producto 7: Fideo Pasta Spaghetti N°5 Bolsa 400 g Acuenta...... -> $460
✅ Producto 8: Fideo Pasta Spaghetti N°5 Bolsa 400 g Acuenta...... -> $460
✅ Producto 9: Fideo Pasta Corbatas Bolsa 400 g Acuenta... -> $460
✅ Producto 10: Fideo Pasta Corbatas Bolsa 400 g Acuenta... -> $460
✅ Producto 11: Fideo Pasta Quifaros Bolsa 400 g Acuenta... -> $460
✅ Producto 12: Fideo Pasta Qu

In [8]:
# --- SCRAPING JUMBO.CL FRUTAS ---
import os
import time
import random
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Entorno limpio.")

# --- VARIABLES ---
NOMBRE_GRUPO = "Vannessa"
CATEGORIA = "Frutas"
URL_JUMBO = "https://www.jumbo.cl/busqueda?ft=frutas"
datos_finales = []

# --- PASO 1: CONFIGURACIÓN ANTI-BOT JUMBO ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-extensions")
options.add_argument("--dns-prefetch-disable")
options.add_argument("--no-first-run")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# JUMBO ES MUY EXIGENTE - MODO HEADLESS NO
driver = None
try:
    driver = webdriver.Chrome(options=options)
    
    # Parches anti-detección avanzados
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
            Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3,4,5]});
            Object.defineProperty(navigator, 'languages', {get: () => ['es-CL','es','en']});
            window.chrome = {runtime: {}};
            Object.defineProperty(navigator, 'permissions', {
                get: () => ({query: () => Promise.resolve({state: 'granted'})})}
            );
        """
    })
    print("🚀 Jumbo anti-bot iniciado.")

    # --- PASO 2: NAVEGACIÓN JUMBO ---
    print(f"🌐 Cargando Jumbo {CATEGORIA}...")
    driver.get(URL_JUMBO)
    
    # Espera EXTREMA para SPA Jumbo (15+ seg)
    print("⏳ Esperando renderizado SPA...")
    time.sleep(15)

    # 🔧 POP-UPS JUMBO (cookies + ubicación)
    def cerrar_popups():
        popups_cerrados = 0
        close_selectores = [
            "//button[contains(text(),'Aceptar') or contains(text(),'Permitir')]",
            "//button[contains(@class,'cookie')]",
            "//span[contains(text(),'×') or text()='✕']",
            "//button[@aria-label='Cerrar']",
            "[data-testid*='close']"
        ]
        
        for selector in close_selectores:
            try:
                buttons = driver.find_elements(By.XPATH, selector)
                for btn in buttons:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        popups_cerrados += 1
                        time.sleep(1)
            except:
                continue
        return popups_cerrados

    popups = cerrar_popups()
    print(f"🍪 {popups} pop-ups cerrados.")
    time.sleep(3)

    # 🔧 SCROLL ULTRA AGRESIVO JUMBO (20 scrolls)
    print("📜 Scroll infinito EXTREMO para Jumbo...")
    last_count = 0
    scroll_count = 0
    
    while scroll_count < 20:
        # Scroll múltiple humano
        for _ in range(3):
            scroll_y = random.randint(700, 1400)
            driver.execute_script(f"window.scrollBy(0, {scroll_y});")
            time.sleep(random.uniform(2.5, 4.5))
        
        # Scroll completo
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(4, 7))
        
        # Contar productos actuales
        productos = len(driver.find_elements(By.CSS_SELECTOR, 
            "[class*='product'],[data-testid*='product'],.product-card"))
        
        print(f"📜 Scroll {scroll_count+1}/20 | Productos: {productos}")
        
        if productos == last_count:
            print("🛑 No más productos nuevos.")
            break
            
        last_count = productos
        scroll_count += 1

    time.sleep(8)  # Espera final

    # 🔍 SELECTORES JUMBO REALES (inspeccionados)
    print("🔍 Buscando frutas Jumbo...")
    
    selectores_jumbo = [
        "[data-testid='product-tile']",
        "div[data-testid*='product']",
        ".product-card",
        "[class*='ProductCard']",
        "[class*='product-item']",
        "article[class*='product']",
        ".sku-card"
    ]
    
    bloques = []
    for selector in selectores_jumbo:
        bloques = driver.find_elements(By.CSS_SELECTOR, selector)
        if len(bloques) > 3:
            print(f"✅ Jumbo encontró {len(bloques)} con: {selector}")
            break

    # FALLBACK JUMBO: buscar contenedores con precio
    if len(bloques) < 5:
        print("🔄 Fallback Jumbo...")
        bloques = driver.find_elements(By.XPATH, 
            "//div[contains(@class,'product') or contains(@class,'sku') or contains(@class,'card')]")
        print(f"📦 {len(bloques)} contenedores fallback")

    # 🍎 EXTRACCIÓN FRUTAS JUMBO
    print(f"🍍 Procesando {min(35, len(bloques))} productos...")
    frutas_validas = 0
    
    palabras_fruta = ['manzana', 'pera', 'plátano', 'banana', 'naranja', 
                     'mandarina', 'uva', 'frutilla', 'fresa', 'kiwi', 
                     'mango', 'piña', 'maracuyá', 'durazno', 'melón', 
                     'sandía', 'papaya', 'fruta', 'limón', 'lime']

    for i, bloque in enumerate(bloques[:35]):
        try:
            # NOMBRE
            nombre = None
            for sel in ["[class*='name']", "[class*='title']", "h3", ".product-name", "[data-testid*='name']"]:
                try:
                    elem = bloque.find_element(By.CSS_SELECTOR, sel)
                    nombre = elem.text.strip()
                    if len(nombre) > 3:
                        break
                except:
                    continue
            
            if not nombre:
                lines = bloque.text.split('\n')
                for line in lines:
                    if len(line) > 5 and any(p in line.lower() for p in palabras_fruta):
                        nombre = line.strip()
                        break

            # PRECIO
            precio = None
            for sel in ["[class*='price']", "[class*='Price']", ".precio", "[data-testid*='price']"]:
                try:
                    elem = bloque.find_element(By.CSS_SELECTOR, sel)
                    precio = elem.text.strip()
                    if '$' in precio:
                        break
                except:
                    continue

            # VALIDAR FRUTA
            if (nombre and precio and 
                any(palabra in nombre.lower() for palabra in palabras_fruta)):
                
                datos_finales.append({
                    "identificador": nombre[:130],
                    "valor": precio,
                    "valor_limpio": 0.0,
                    "categoria": CATEGORIA,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Jumbo",
                    "url": URL_JUMBO,
                    "posicion": frutas_validas + 1
                })
                frutas_validas += 1
                print(f"🍎 [{frutas_validas:2d}] {nombre[:60]:<60} | {precio}")

        except Exception as e:
            continue

    print(f"\n🎉 Jumbo: {frutas_validas} frutas válidas!")

except Exception as e:
    print(f"❌ Error Jumbo: {e}")
finally:
    if driver:
        driver.quit()

# --- LIMPIEZA Y GUARDADO ---
if datos_finales:
    print("\n💰 Limpiando precios Jumbo...")
    for d in datos_finales:
        # Jumbo: $1.234 o $ 1.234,56
        raw = re.sub(r'[^\d,.]', '', d["valor"])
        try:
            if ',' in raw:
                raw = raw.replace(',', '.')
            d["valor_limpio"] = float(raw)
        except:
            d["valor_limpio"] = 0.0

    # MongoDB
    try:
        client = MongoClient("mongodb", 27017)
        db = client["Canasta_db"]
        coleccion = db["Retail_A"]
        coleccion.insert_many(datos_finales)
        print(f"💾 ✅ {len(datos_finales)} frutas Jumbo guardadas.")
    except Exception as e:
        print(f"MongoDB: {e}")

    # JSON + RESUMEN
    import json
    with open("jumbo_frutas.json", "w", encoding="utf-8") as f:
        json.dump(datos_finales, f, ensure_ascii=False, indent=2)
    
    print("\n🍍 TOP FRUTAS JUMBO:")
    for d in sorted(datos_finales[:12], key=lambda x: x.get('valor_limpio', 0)):
        print(f"  {d['identificador'][:65]:<65} | ${d['valor_limpio']:>9,.0f}")

else:
    print("⚠️ Sin frutas. Verifica CAPTCHA o pop-ups.")

🧹 Entorno limpio.
🚀 Jumbo anti-bot iniciado.
🌐 Cargando Jumbo Frutas...
⏳ Esperando renderizado SPA...
🍪 0 pop-ups cerrados.
📜 Scroll infinito EXTREMO para Jumbo...
📜 Scroll 1/20 | Productos: 237
📜 Scroll 2/20 | Productos: 237
🛑 No más productos nuevos.
🔍 Buscando frutas Jumbo...
🔄 Fallback Jumbo...
📦 198 contenedores fallback
🍍 Procesando 35 productos...

🎉 Jumbo: 0 frutas válidas!
⚠️ Sin frutas. Verifica CAPTCHA o pop-ups.


In [4]:
# --- SCRAPER CANASTA BÁSICA ACUENTA.CL ---
import os
import time
import random
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("🧹 Scraper Canasta Básica iniciado")

# VARIABLES
URL_CANASTA = "https://www.acuenta.cl/search?name=canasta%20basica"
NOMBRE_GRUPO = "Canasta Básica Acuenta"
datos_finales = []

# Configuración anti-detección
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = None
try:
    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    print("🚀 Navegador listo")

    # 🌐 CARGAR CANASTA BÁSICA
    print(f"🛍️ Cargando CANASTA BÁSICA: {URL_CANASTA}")
    driver.get(URL_CANASTA)
    time.sleep(10)

    # Cookies
    try:
        cookie_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Aceptar') or contains(text(), 'Acepto') or contains(@class, 'cookie')]"))
        )
        cookie_btn.click()
        print("🍪 Cookies aceptadas")
        time.sleep(2)
    except:
        print("ℹ️ Sin cookies")

    # 📜 SCROLL AGRESIVO para cargar TODA la canasta
    print("📜 Scroll para cargar TODOS los productos...")
    last_height = driver.execute_script("return document.body.scrollHeight")
    scroll_count = 0
    
    while scroll_count < 20:  # Máximo 20 scrolls
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(2, 4))
        new_height = driver.execute_script("return document.body.scrollHeight")
        
        if new_height == last_height:
            time.sleep(3)
            if driver.execute_script("return document.body.scrollHeight") == new_height:
                break
        last_height = new_height
        scroll_count += 1
        print(f"Scroll {scroll_count}/20 - Altura: {new_height:,}")
    
    time.sleep(4)

    # 🔍 SELECTORES OPTIMIZADOS CANASTA BÁSICA
    print("🔍 Buscando productos CANASTA BÁSICA...")
    
    selectores_canasta = [
        "div[class*='product']", "div[class*='ProductCard']",
        "div[class*='product-card']", "div[class*='card']",
        ".product-item", "[data-testid*='product']",
        "article[class*='product']", ".grid-item"
    ]
    
    bloques = []
    for selector in selectores_canasta:
        try:
            bloques = driver.find_elements(By.CSS_SELECTOR, selector)
            if len(bloques) > 3:
                print(f"✅ Canasta: {len(bloques)} productos con '{selector}'")
                break
        except:
            continue

    # 🔧 Fallback: buscar por precios
    if not bloques or len(bloques) < 5:
        print("🔍 Fallback: buscando por PRECIOS...")
        bloques = driver.find_elements(By.XPATH, "//*[contains(text(), '$')]/ancestor::div[2]")
        bloques = list(set(bloques))[:40]
        print(f"✅ {len(bloques)} productos por precio")

    if not bloques:
        print("🔧 DEBUG Canasta:")
        print("Título:", driver.title)
        print(driver.page_source[:3000])
        exit()

    # 📦 EXTRACCIÓN CANASTA BÁSICA
    print(f"📦 Procesando {len(bloques)} productos de CANASTA BÁSICA...")
    
    for i, bloque in enumerate(bloques[:40]):  # Hasta 40 productos
        try:
            # NOMBRE
            nombre = None
            selectores_nombre = [
                "[class*='name']", "[class*='title']", "h3", "h2", 
                ".product-name", ".titulo", "[data-testid*='name']"
            ]
            
            for sel in selectores_nombre:
                try:
                    elem = bloque.find_element(By.CSS_SELECTOR, sel)
                    nombre_cand = elem.text.strip()
                    if len(nombre_cand) > 3 and len(nombre_cand) < 100:
                        nombre = nombre_cand
                        break
                except:
                    continue
            
            # Fallback nombre
            if not nombre:
                texto = bloque.text.strip()
                lineas = [l for l in texto.split('\n') if l.strip()]
                for linea in lineas:
                    if '$' not in linea and len(linea) > 3:
                        nombre = linea[:80]
                        break
            
            # PRECIO
            precio = None
            selectores_precio = [
                "[class*='price']", "[class*='precio']", ".price",
                "span[class*='price']", "[data-testid*='price']"
            ]
            
            for sel in selectores_precio:
                try:
                    elem = bloque.find_element(By.CSS_SELECTOR, sel)
                    precio_cand = elem.text.strip()
                    if '$' in precio_cand:
                        precio = precio_cand
                        break
                except:
                    continue
            
            # Fallback precio
            if not precio:
                match = re.search(r'\$[\d,\.\s]+', bloque.text)
                precio = match.group() if match else None
            
            # ✅ GUARDAR
            if nombre and precio:
                datos_finales.append({
                    "identificador": nombre[:80],
                    "valor": precio,
                    "valor_limpio": 0.0,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Acuenta",
                    "url": URL_CANASTA,
                    "posicion": i+1,
                    "categoria": "Canasta Básica"
                })
                print(f"✅ [{i+1:2d}] {nombre[:45]:<45} -> {precio}")
                
        except Exception as e:
            print(f"⚠️ Error {i+1}: {str(e)[:40]}")
            continue

    print(f"\n🎉 CANASTA BÁSICA: {len(datos_finales)} productos")

except Exception as e:
    print(f"❌ Error: {e}")
finally:
    if driver:
        driver.quit()

# 💰 LIMPIEZA Y TABLA CANASTA BÁSICA
if datos_finales:
    print("\n💰 Limpiando precios canasta...")
    precios_limpios = []
    
    for d in datos_finales:
        raw = re.sub(r'[^\d,.]', '', d["valor"])
        numeros = re.findall(r'\d+[.,]?\d*', raw)
        if numeros:
            precio = float(numeros[0].replace(",", "."))
            if precio > 0:
                d["valor_limpio"] = precio
                precios_limpios.append(precio)
    
    # 📊 TABLA CANASTA BÁSICA PROFESIONAL
    print("\n" + "═"*95)
    print("🛍️  CANASTA BÁSICA ACUENTA.CL")
    print(f"🔗 {URL_CANASTA}")
    print("═"*95)
    print(f"{'PRODUCTO':<50} {'PRECIO':>12} {'POS':>5}")
    print("─"*95)
    
    for i, d in enumerate(datos_finales, 1):
        if d["valor_limpio"] > 0:
            print(f"{d['identificador'][:50]:<50} ${d['valor_limpio']:>10,.0f} {d['posicion']:>5}")
    
    if precios_limpios:
        total_canasta = sum(precios_limpios)
        promedio = total_canasta / len(precios_limpios)
        print("─"*95)
        print(f"{'TOTAL PRODUCTOS:':<50} {len(precios_limpios):>12} {'':>5}")
        print(f"{'COSTO TOTAL CANASTA:':<50} ${total_canasta:>10,.0f} {'':>5}")
        print(f"{'PROMEDIO POR PRODUCTO:':<50} ${promedio:>10,.0f} {'':>5}")
        print(f"{'PRODUCTO MÁS BARATO:':<50} ${min(precios_limpios):>10,.0f} {'':>5}")
        print(f"{'PRODUCTO MÁS CARO:':<50} ${max(precios_limpios):>10,.0f} {'':>5}")
        print("═"*95)
    
    # 💾 EXPORTAR
    import json
    with open("canasta_basica_acuenta.json", "w", encoding="utf-8") as f:
        json.dump(datos_finales, f, ensure_ascii=False, indent=2)
    print("💾 ✅ canasta_basica_acuenta.json")
    
    # Resumen TXT
    with open("resumen_canasta_basica.txt", "w", encoding="utf-8") as f:
        f.write(f"CANASTA BÁSICA ACUENTA.CL\n")
        f.write(f"URL: {URL_CANASTA}\n")
        f.write(f"Fecha: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"TOTAL: {len(precios_limpios)} productos\n")
        f.write(f"COSTO TOTAL: ${total_canasta:,.0f}\n")
        f.write(f"PROMEDIO: ${promedio:,.0f}\n\n")
        f.write("PRODUCTOS:\n")
        f.write("-"*50 + "\n")
        for d in datos_finales:
            if d["valor_limpio"] > 0:
                f.write(f"${d['valor_limpio']:>8,.0f} - {d['identificador']}\n")
    print("💾 ✅ resumen_canasta_basica.txt")
    
    # MongoDB
    try:
        client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
        db = client["Canasta_db"]
        db["Canasta_Basica_Acuenta"].insert_many(datos_finales)
        print("💾 ✅ MongoDB Canasta Básica")
    except:
        print("⚠️ Sin MongoDB")

else:
    print("❌ No se encontraron productos de canasta básica")

🧹 Scraper Canasta Básica iniciado
🚀 Navegador listo
🛍️ Cargando CANASTA BÁSICA: https://www.acuenta.cl/search?name=canasta%20basica
ℹ️ Sin cookies
📜 Scroll para cargar TODOS los productos...
Scroll 1/20 - Altura: 1,741
🔍 Buscando productos CANASTA BÁSICA...
✅ Canasta: 42 productos con 'div[class*='product']'
📦 Procesando 42 productos de CANASTA BÁSICA...
✅ [ 1] Toalla de Papel Nova 14m 2 Un Nova            -> $1.000
✅ [ 3] Toalla de Papel Nova 14m 2 Un Nova            -> $1.000
✅ [ 4] Toalla de Papel Nova 14m 2 Un Nova            -> $1.000
✅ [ 5] Plátano Granel 500 g (3 a 4 un aprox) Multi M -> $695
✅ [ 6] Plátano Granel 500 g (3 a 4 un aprox) Multi M -> $695
✅ [ 7] Yoghurt Batido Sabor Plátano Pote 120 g Sopro -> $270
✅ [ 8] Yoghurt Batido Sabor Plátano Pote 120 g Sopro -> $270
✅ [ 9] Jurel Natural Lata Drenado 300 g - Neto 425 g -> $1.990
✅ [10] Jurel Natural Lata Drenado 300 g - Neto 425 g -> $1.990
✅ [11] Leche en Polvo Entera Bolsa 900 g Colun       -> $7.150
✅ [12] Leche en Polvo